# Thành viên 2 — Kaggle `GPU T4 x 2` (32 GB VRAM): Track Quốc tế `PHEME` (`Twitter-RoBERTa + mDeBERTa Stance Verbalizer -> SBi-FAGAT`)

**Cấu hình bắt buộc trên Kaggle trước khi chạy song song với Thành viên 1:**
1. Bật **Accelerator: GPU T4 x 2** (tổng 32 GB VRAM).
2. Bật **Internet: ON** để tải `PHEME` từ Figshare (có kiểm tra MD5) và trọng số Hugging Face.

Bạn có 2 chế độ chạy song song:
- **`member2_en`** *(Khuyên dùng để chia 50/50 với Thành viên 1)*: Chạy trọn gói dữ liệu chuẩn quốc tế `PHEME` (`cardiffnlp/twitter-roberta-base-offensive` + `mDeBERTa-v3-xnli` Stance Verbalizer -> chưng cất Cross-Encoder 2D `cardiffnlp/twitter-roberta-base` -> trích xuất `.pt` *Ancestor SpanPool* -> 5-seed công bằng Inductive + Leave-One-Event-Out + Ablation).
- **`model_gnn_only`**: Chuyên trách train & tinh chỉnh **Mạng Đồ thị `SBi-FAGAT` + 7 mô hình so sánh** (nếu đã nạp `processed/vn` từ Thành viên 1 hoặc bật `--bootstrap-if-missing` để chạy ngay không cần chờ).

In [ ]:
# Bước 1: Kiểm tra GPU NVIDIA Tesla T4
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available(), "| Num GPUs:", torch.cuda.device_count())

In [ ]:
# Bước 2: Clone/cập nhật repo từ GitHub & cài gói
%cd /kaggle/working
!if [ ! -d "gnn-nlp-propagation" ]; then git clone https://github.com/hieu-kiien/gnn-nlp-propagation.git; fi
%cd /kaggle/working/gnn-nlp-propagation
!git pull origin main
!pip install -q -r requirements.txt
!pytest -q

In [ ]:
# Bước 3: Kiểm thử khói toàn bộ đường ống (xong trong <10 giây)
!python -m scripts.kaggle_parallel_runner --role member2_en --smoke-test

In [ ]:
# Bước 4: CHẠY CHÍNH THỨC SONG SONG TRÊN KAGGLE
# Đổi ROLE = "model_gnn_only" và thêm --bootstrap-if-missing nếu bạn chỉ muốn chạy nhánh GNN
ROLE = "member2_en"  # hoặc "model_gnn_only"

!python -m scripts.kaggle_parallel_runner \
    --role {ROLE} \
    --lang en \
    --hidden-dim 64 \
    --epochs-nlp 3 \
    --epochs-gnn 100

In [ ]:
# Bước 5 (Tùy chọn cho bài báo quốc tế): Chạy thêm thực nghiệm mở rộng UPFD GossipCop
!python -m scripts.run_upfd_extension --dataset-name gossipcop --feature bert --hidden-dim 64

## Xuất kết quả gộp với Thành viên 1
Trước khi tắt máy ảo Kaggle, tải **2 gói nén** trong `/kaggle/working/gnn-nlp-propagation/`:
1. `sbifagat_member2_en_bundle.zip` — gồm `paper/results_main_pheme.json`, `paper/results_loo_pheme.json`, `paper/results_ablation_pheme.json`, `paper/tables_*_pheme.tex`, `checkpoints/best_sbifagat_pheme.pt`, `paper/figures/pheme/*.pdf` và phiếu chấm mù `data/interim/human_audit_en_300.csv`.
2. `sbifagat_member2_en_sealed_machine_keys.zip` — gói niêm phong riêng chứa `data/interim/human_audit_en_300_machine_key.csv` (giữ riêng để đối chiếu `machine_vs_gold` sau khi hoàn tất chấm mù).

*Lưu ý:* Ở chế độ mặc định công bằng không chạy `auto_loop` (thêm `--run-auto-loop` ở Bước 4 nếu muốn xuất thêm `outputs/auto_loop_pheme/{loop_diagnostics,worst_val_cases_for_review}.json`).